# Prueba hacia adelante: la semana que el modelo no vio

La tercera presentación pide mostrar el mejor modelo, compararlo con los demás y dar ejemplos concretos de éxito y de mejora. Aquí se hace con **datos que no estaban en la presentación anterior**: se predice una semana completa (lunes a domingo) de octubre de 2026.

**Qué se compara** (mismas variables y mismos hiperparámetros que `06` y `10`, versión 1 de `busqueda_hiperparametros.csv`):

| Modelo | Qué es |
|---|---|
| XGBoost congelado | Entrenado solo con lo que había en la presentación anterior (hasta `CORTE_CONGELADO`). Dice cómo se comporta con datos nuevos sin tocarlo. |
| XGBoost reentrenado | Entrenado con todo lo anterior al primer día de la semana de prueba. Dice cuánto mejora al actualizarlo. |
| Promedio histórico, media de 4 semanas, regresión lineal | Las referencias de siempre. |
| MLP (opcional) | Con `USAR_MLP = True`. Tarda más; en Colab con GPU se acelera la parte de XGBoost. |

**Por qué no hay fuga de datos.** Todas las variables de rezago tienen al menos 7 días de antigüedad. Para el último día de la semana de prueba (inicio + 6) el rezago más reciente es el día anterior al inicio de la semana, así que ninguna variable usa datos de la semana que se predice. La prueba equivale a predecir con una semana de anticipación.

**Cómo correrlo:** pon `2026-10.csv` en `data/raw/` y Restart & Run All. Con `INICIO_PRUEBA` y `CORTE_CONGELADO` se puede ensayar con otra semana (ver abajo).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
import modelo as M
import mismo_mes as MM
import ultima_semana as U

# --- Semana de prueba (lunes a domingo) y versión congelada
INICIO_PRUEBA = "2026-10-19"      # lunes; 19 al 25 de octubre
DIAS = 7
CORTE_CONGELADO = "2026-09-30"    # último día de datos de la presentación anterior
DESDE = "2025-09"                 # primer mes de historia (pon "2026-03" si te falta memoria)
USAR_MLP = False
ETIQUETA = "oct"                  # sufijo de los archivos de salida

# Para ensayar con septiembre (antes de tener octubre):
# INICIO_PRUEBA, CORTE_CONGELADO, ETIQUETA = "2026-09-21", "2026-08-31", "ensayo_sep"

FIG = "../figures"
PROC = M.PROCESSED_DIR
os.makedirs(FIG, exist_ok=True)

ini = pd.Timestamp(INICIO_PRUEBA)
fin = ini + pd.Timedelta(days=DIAS - 1)
assert ini.dayofweek == 0, "INICIO_PRUEBA debe ser lunes"
print(f"Prueba: {ini:%d %b %Y} a {fin:%d %b %Y} | modelo congelado hasta {CORTE_CONGELADO}")

In [ ]:
faltan = U.revisar(INICIO_PRUEBA, DIAS, DESDE)
if faltan:
    raise FileNotFoundError(
        "Faltan los archivos de: " + ", ".join(faltan) + ". Ponlos en data/raw/ con el nombre AAAA-MM.csv "
        "(por ejemplo 2026-10.csv). Si solo falta historia vieja, sube DESDE (por ejemplo '2026-03').")

print("Dispositivo XGBoost:", MM.configurar_dispositivo())     # usa la GPU si hay una (Colab)

estaciones = M.cargar_estaciones()
h = pd.read_csv(f"{PROC}/busqueda_hiperparametros.csv").iloc[0]
params = MM.cargar_params()
print("Hiperparámetros (versión 1):", params)

tabla = U.preparar_tabla(INICIO_PRUEBA, DIAS, DESDE, estaciones)
cong, reent, test = U.partir(tabla, INICIO_PRUEBA, DIAS, CORTE_CONGELADO)
assert reent["fecha"].max() < ini, "el entrenamiento no debe tocar la semana de prueba"
print(f"Entrenamiento congelado: {len(cong):,} filas hasta {cong['fecha'].max().date()}")
print(f"Entrenamiento reentrenado: {len(reent):,} filas hasta {reent['fecha'].max().date()}")
print(f"Prueba: {len(test):,} filas, {test['estacion'].nunique()} estaciones, {test['fecha'].nunique()} días")

## 1. Entrenar y predecir

In [ ]:
pred = U.predecir(cong, reent, test, params, usar_mlp=USAR_MLP)
modelos = U.columnas_modelo(pred)
print(modelos)

## 2. Comparación de modelos

RMSE y MAE en salidas por estación-hora; `Sesgo` es predicho menos real (positivo = predice de más) y `Sesgo_%_de_demanda` lo expresa como porcentaje de la demanda media de la semana.

In [ ]:
res = U.tabla_metricas(pred)
res.insert(0, "semana_inicio", INICIO_PRUEBA)
res.round(4).to_csv(f"{PROC}/resultados_ultima_semana_{ETIQUETA}.csv", index=False)
res.round(3)

In [ ]:
color = {"XGBoost reentrenado": "#0f7a62", "XGBoost congelado": "#6bb8a4", "MLP": "#8a3fb5"}
orden = res.sort_values("RMSE", ascending=False)
fig, ax = plt.subplots(figsize=(8, 0.6 * len(orden) + 1.2))
ax.barh(orden["modelo"], orden["RMSE"], color=[color.get(m, "#9aa3b2") for m in orden["modelo"]])
for y, v in enumerate(orden["RMSE"]):
    ax.text(v, y, f" {v:.3f}", va="center")
ax.set_xlabel("RMSE (salidas por estación-hora)")
ax.set_title(f"Semana del {ini:%d/%m} al {fin:%d/%m}")
ax.set_xlim(0, orden["RMSE"].max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIG}/25_ultima_semana_rmse_{ETIQUETA}.png", dpi=150)
plt.show()

### Error por día

In [ ]:
por_dia = U.metricas_por_dia(pred)
por_dia.index = [f"{d:%a %d/%m}" for d in por_dia.index]
por_dia.round(2)

## 3. Ejemplos: la red completa, hora por hora

Suma de salidas de **todas** las estaciones, real contra predicho. Sirve para ver si el modelo sigue el ritmo de la semana (picos de entre semana, fin de semana) y dónde se separa de lo real.

In [ ]:
mostrar = [m for m in ("XGBoost reentrenado", "XGBoost congelado") if m in pred.columns]
red = U.red_por_hora(pred, mostrar)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(red.index, red[M.TARGET], color="black", lw=1.8, label="Real")
ax.plot(red.index, red["XGBoost reentrenado"], color="#0f7a62", lw=1.5, label="XGBoost reentrenado")
if "XGBoost congelado" in red:
    ax.plot(red.index, red["XGBoost congelado"], color="#6bb8a4", lw=1.2, ls="--", label="XGBoost congelado")
ax.set_xticks(pd.date_range(ini, fin, freq="D"))
ax.set_xticklabels([f"{d:%a %d/%m}" for d in pd.date_range(ini, fin, freq="D")])
ax.set_ylabel("Salidas en toda la red por hora")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIG}/26_ultima_semana_red_{ETIQUETA}.png", dpi=150)
plt.show()

tot = pred.groupby("fecha")[[M.TARGET, "XGBoost reentrenado"]].sum()
tot["error_%"] = 100 * (tot["XGBoost reentrenado"] - tot[M.TARGET]) / tot[M.TARGET]
tot.index = [f"{d:%a %d/%m}" for d in tot.index]
tot.round(1)

## 4. Sesgo por hora

Predicho menos real, por hora del día. En `09` y `10` el modelo predecía de más de 19 a 23 h. Aquí se ve si el patrón se repite en una semana nueva.

In [ ]:
sh = U.sesgo_por_hora(pred, [m for m in ("XGBoost reentrenado", "XGBoost congelado", "Promedio histórico") if m in pred.columns])
fig, ax = plt.subplots(figsize=(9, 3.8))
for m, col, ls in (("XGBoost reentrenado", "#0f7a62", "-"), ("XGBoost congelado", "#6bb8a4", "--"), ("Promedio histórico", "#9aa3b2", ":")):
    if m in sh:
        ax.plot(sh.index, sh[m], color=col, ls=ls, lw=1.8, label=m)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Sesgo (predicho − real)")
ax.set_xticks(range(0, 24, 2))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{FIG}/27_ultima_semana_sesgo_hora_{ETIQUETA}.png", dpi=150)
plt.show()

## 5. Ejemplos concretos: estaciones donde acierta y donde falla

Se consideran las estaciones con al menos una salida por hora en promedio. **Casos de éxito**: menor error relativo (error absoluto medio entre demanda media). **Áreas de mejora**: mayor error relativo. El modelo usado es `MODELO_EJ`.

In [ ]:
MODELO_EJ = "XGBoost reentrenado"
est = U.resumen_estaciones(pred, MODELO_EJ, min_demanda=1.0)
print(f"{len(est)} estaciones con demanda media >= 1 salida por hora")
buenas, malas = est.head(3), est.tail(3)
pd.concat({"éxito": buenas, "mejora": malas}).round(2)

In [ ]:
def serie(est_id):
    d = pred[pred["estacion"] == est_id].sort_values(["fecha", "hora"])
    return d["fecha"] + pd.to_timedelta(d["hora"], unit="h"), d

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, (est_id, fila), tipo in zip(axes.flat, list(buenas.iterrows()) + list(malas.iterrows()), ["Éxito"] * 3 + ["Mejora"] * 3):
    x, d = serie(est_id)
    ax.plot(x, d[M.TARGET], color="black", lw=1.2, label="Real")
    ax.plot(x, d[MODELO_EJ], color="#0f7a62", lw=1.2, label=MODELO_EJ)
    ax.set_title(f"{tipo}: estación {est_id} (MAE/demanda {fila['error_relativo_%']:.0f} %)", fontsize=10)
    ax.set_xticks(pd.date_range(ini, fin, freq="2D"))
    ax.set_xticklabels([f"{d_:%d/%m}" for d_ in pd.date_range(ini, fin, freq="2D")])
    ax.spines[["top", "right"]].set_visible(False)
axes.flat[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG}/28_ultima_semana_ejemplos_{ETIQUETA}.png", dpi=150)
plt.show()

### Casos concretos de estación-hora

Cinco aciertos y cinco fallos entre las filas con al menos 5 salidas reales. Los fallos más grandes suelen ser ráfagas de demanda que ningún rezago anticipa; conviene revisar si son estaciones combinadas (ids como `273-274`) o días especiales.

In [ ]:
casos = U.casos_estacion_hora(pred, MODELO_EJ, n=5, min_real=5)
print("Aciertos (error cercano a cero):")
display(casos["aciertos"].round(2))
print("Fallos (mayor error absoluto):")
display(casos["fallos"].round(2))

## 6. Guardar las predicciones

In [ ]:
pred.round(3).to_csv(f"{PROC}/predicciones_ultima_semana_{ETIQUETA}.csv", index=False)
print("Guardado:", f"{PROC}/predicciones_ultima_semana_{ETIQUETA}.csv", f"({len(pred):,} filas)")

## Lectura de resultados

*(Completar después de correr con octubre. Responde, con los números de arriba:)*

- **Mejor modelo.** ¿Cuál tiene el menor RMSE en la semana nueva? ¿Es el mismo que en las validaciones anteriores? ¿Qué tan lejos queda del promedio histórico y de la media de 4 semanas?
- **Congelado vs reentrenado.** ¿Reentrenar con las semanas más recientes mejora algo? Si no, ¿qué dice eso de la estabilidad del patrón?
- **Sesgo por hora.** ¿Reaparece la sobreestimación de 19 a 23 h? ¿Y la subestimación del pico de la mañana?
- **Éxitos y mejoras.** ¿Qué tienen en común las estaciones donde acierta y donde falla (volumen, ubicación, estaciones combinadas)?
- **Limitación.** Una sola semana es una muestra corta; el resultado no es una garantía para otros periodos.